In [1]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

sys.path.append("..")
from src import create_sentence_nace_code_similarities, analysis_functions
import test_base
from sentence_splitter import split_text_into_sentences

/Users/hendrikweichel/miniconda3/envs/nace_project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Test retrieving the similarities for chunks in a pdf to the NACE Code

**Function:** pdf-> (chunk x code -> [-1,1])

**Parameters:** 

- pdf_path
- way of chunking the text (e.g. sentences, sliding window, or paragraphs)
- way of preprocessing (most is fixed for all reports)
    - similarity threshold of relevant chunks
    - length of irrelevant chunks

**Store analytics for each datapoint:**

- mean score for each class given a threshold

In [2]:
# Parameters: 

threshold_min_chunk_len = 100
cos_threshold = 0.4
sentence_length = 6

In [3]:
dataset_path = "../data/datasets/german_annual_reports"
dataset_path = "../data/datasets/stoxx_600"
dataset_path = "../data/datasets/stoxx_600_extended"
dataset_path = "../data/datasets/reports_subset_from_full_data_1"

In [4]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [5]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0)
nace_classes.head()

,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,NACE,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,1.19,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,1.41,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,2.30,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,1.19,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,1.19,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf


In [6]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'Ariston Holdings Ltd.1.pdf': 1.19,
 'Heritage Foods Limited1.pdf': 1.41,
 'Timberwell Bhd.1.pdf': 2.3,
 'Matang Bhd.2.pdf': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.pdf': 1.19,
 'Tech-bank Food Co., Ltd.3.pdf': 1.46,
 'Namoi Cotton Ltd1.pdf': 1.63,
 'Atlantic Sapphire ASA1.pdf': 3.11,
 'Agra Limited2.pdf': 1.19,
 'Huisheng International Holdings Ltd.3.pdf': 1.46,
 'Australian Agricultural Company Limited1.pdf': 1.62,
 'Waterbase Limited2.pdf': 3.21,
 'CannAmerica Brands Corp.1.pdf': 1.3,
 'Qian Hu Corporation Limited1.pdf': 3.21,
 'Jawala Inc.1.pdf': 1.19,
 'Green Thumb Industries Inc.1.pdf': 1.19,
 'CLS Holdings USA Inc2.pdf': 1.19,
 'China Bozza Development Holdings Limited1.pdf': 2.4,
 'Salmon Evolution ASA1.pdf': 3.21,
 'North American Cannabis Holdings, Inc.1.pdf': 1.19,
 'Malwatte Valley Plantations Plc1.pdf': 1.61,
 'Greenheart Group Limited1.pdf': 2.2,
 'Bumitama Agri Ltd.1.pdf': 1.19,
 'Genus plc1.pdf': 1.62,
 'Kotagala Plantations Plc1.pdf': 2.3,
 'PT Andira Agro Tbk1.pdf': 1.1

In [7]:
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
report_to_nace_class

{'Ariston Holdings Ltd.1.txt': 1.19,
 'Heritage Foods Limited1.txt': 1.41,
 'Timberwell Bhd.1.txt': 2.3,
 'Matang Bhd.2.txt': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.txt': 1.19,
 'Tech-bank Food Co., Ltd.3.txt': 1.46,
 'Namoi Cotton Ltd1.txt': 1.63,
 'Atlantic Sapphire ASA1.txt': 3.11,
 'Agra Limited2.txt': 1.19,
 'Huisheng International Holdings Ltd.3.txt': 1.46,
 'Australian Agricultural Company Limited1.txt': 1.62,
 'Waterbase Limited2.txt': 3.21,
 'CannAmerica Brands Corp.1.txt': 1.3,
 'Qian Hu Corporation Limited1.txt': 3.21,
 'Jawala Inc.1.txt': 1.19,
 'Green Thumb Industries Inc.1.txt': 1.19,
 'CLS Holdings USA Inc2.txt': 1.19,
 'China Bozza Development Holdings Limited1.txt': 2.4,
 'Salmon Evolution ASA1.txt': 3.21,
 'North American Cannabis Holdings, Inc.1.txt': 1.19,
 'Malwatte Valley Plantations Plc1.txt': 1.61,
 'Greenheart Group Limited1.txt': 2.2,
 'Bumitama Agri Ltd.1.txt': 1.19,
 'Genus plc1.txt': 1.62,
 'Kotagala Plantations Plc1.txt': 2.3,
 'PT Andira Agro Tbk1.txt': 1.1

In [8]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
reports_path

['../data/datasets/reports_subset_from_full_data_1/TXTs/PVH Corp.3.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Mekdam Holding Group Company1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Ambea AB1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Harbour Equine Holdings Limited3.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Poste Italiane SpA1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Propel Funeral Partners Ltd.1.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Nabors Industries Ltd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/MK Land Holdings Bhd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Brookfield Infrastructure Corp. (New York)2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Shangri-La Hotel Public Co. Ltd.2.txt',
 '../data/datasets/reports_subset_from_full_data_1/TXTs/Rentokil Initial plc1.txt',
 '../data/datasets/reports_subset_fro

In [9]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [11]:
for i in range(4,5): 
    nace_level = i

    result_path = f"../results/dataset__{dataset_name}_sentence_len_{sentence_length}__min_chunk_len_{threshold_min_chunk_len}__cos_thresh_{cos_threshold}__nace_level_{nace_level}"

    res = test_base.test_report_classification(
        reports_path=reports_path, 
        preprocess_report=preprocess_report, 
        report_to_nace_class=report_to_nace_class, 
        result_path = result_path, 
        threshold_min_chunk_len=threshold_min_chunk_len, 
        cos_threshold=cos_threshold,  
        level=i, 
        overwrite=True, 
        classification_function=create_sentence_nace_code_similarities.classification_by_similarities)

  0%|          | 0/1555 [00:00<?, ?it/s]

 84%|████████▍ | 1308/1555 [47:33<6:12:14, 90.42s/it] 2025-11-14 12:37:17.229 python[60557:1514466] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-60557-2025-11-14_12_37_16-2904892647‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
 84%|████████▍ | 1313/1555 [55:10<6:14:01, 92.74s/it] 2025-11-14 12:44:45.283 python[60557:1514466] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-60557-2025-11-14_12_44_45-4251761780‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
 85%|████████▌ | 1326/1555 [1:10:36<4:14:50, 66.77s/it]2025-11-14 13:00:08.518 python[60557:1514466] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-60557-2025-11-14_13_00_08-3956475666‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
2025-11-14 13:00:08.522 python[60557:1514466] Error creating directory 
 The v

OSError: [Errno 28] No space left on device: '../results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4/EC World Real Estate Investment Trust1.txt/relevant_sentences_EC World Real Estate Investment Trust1.txt'